# Sesión 01 en vivo — LLMs y Setup de Entorno de Trabajo

**Programa AI Engineering — LIDR**

Este notebook contiene todos los ejercicios de la sesión en vivo. Ejecuta las celdas de configuración primero y luego ve avanzando bloque por bloque con el instructor.

---

## Configuración inicial

Ejecuta estas tres celdas antes de empezar. Necesitas tener tus API keys configuradas en **Colab Secrets** (icono de llave 🔑 en el panel lateral):
- `OPENAI_API_KEY` → tu key de OpenAI (empieza por `sk-`)
- `ANTHROPIC_API_KEY` → tu key de Anthropic (empieza por `sk-ant-`)

In [4]:
# Install dependencies
# Dependency warnings from Colab pre-installed packages (gradio, typer) are normal and do not affect our code
# En local las dependencias ya están en el proyecto (uv add); no hace falta pip install

# Verify installation
import openai, anthropic, tiktoken, litellm
from importlib.metadata import version
print(f"✅ openai {openai.__version__}")
print(f"✅ anthropic {anthropic.__version__}")
print(f"✅ tiktoken {version('tiktoken')}")
print(f"✅ litellm {version('litellm')}")

✅ openai 3.52.0
✅ anthropic 2.9.0
✅ tiktoken 0.14.0
✅ litellm 1.109.1


In [5]:
# Load API keys from Colab Secrets
import os
from dotenv import load_dotenv

load_dotenv()  # en local: lee las keys del .env de la raíz (en Colab era userdata.get)

# Create clients
from openai import OpenAI
from anthropic import Anthropic

openai_client = OpenAI()
anthropic_client = Anthropic()

print("✅ Clients configured successfully")

✅ Clients configured successfully


In [6]:
# Utility function to calculate costs (used throughout the session)

PRICING = {
    # OpenAI (USD per 1M tokens)
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o-mini-2024-07-18": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "gpt-5.4-nano": {"input": 0.20, "output": 1.25},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "output": 0.40},
    # Anthropic (USD per 1M tokens)
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6-20250514": {"input": 3.00, "output": 15.00},
}


def calculate_cost(model, input_tokens, output_tokens):
    """Calculate the cost of an API call in USD."""
    prices = PRICING.get(model)
    if not prices:
        for key in PRICING:
            if key in model or model in key:
                prices = PRICING[key]
                break
    if not prices:
        return None

    cost_in = (input_tokens / 1_000_000) * prices["input"]
    cost_out = (output_tokens / 1_000_000) * prices["output"]
    return {"input": cost_in, "output": cost_out, "total": cost_in + cost_out}


print("✅ Cost calculation function loaded")

✅ Cost calculation function loaded


---
# Bloque 2 — Del prompt del usuario al prompt del producto

**Objetivo:** La calidad de la respuesta no puede depender de que el usuario sepa hacer prompting. El desarrollador es quien diseña la experiencia.

### 2.1. El problema: la respuesta genérica

Enviamos una pregunta realista pero sin system prompt. ¿Pondríais esta respuesta en un producto?

In [7]:
# No system prompt — the model has no context about what we need
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="We're migrating our payment service from a monolith to microservices. What should we consider?"
)

print(response.output_text)
print(f"\n--- {response.usage.input_tokens} input tokens + {response.usage.output_tokens} output tokens ---")

Migrating from a monolithic architecture to microservices can be a complex but rewarding process. Here are several key considerations to keep in mind during this transition, particularly for a payment service:

### 1. **Service Decomposition**
   - **Identify Boundaries**: Clearly define the boundaries of each microservice. For a payment service, consider separating functionalities like payment processing, fraud detection, transaction history, and notifications.
   - **Domain-Driven Design**: Use domain-driven design principles to ensure services align with business capabilities.

### 2. **Data Management**
   - **Database Per Service**: Each microservice should ideally have its own database to enforce autonomy and scalability.
   - **Data Consistency**: Establish patterns for managing data consistency across services (event sourcing, CQRS, etc.).
   - **Querying Data**: Plan how services will query data. Consider using API gateways or GraphQL for aggregating data.

### 3. **Communicat

### 2.2. El system prompt como contrato de calidad

Vamos a enviar la **misma pregunta** con system prompts cada vez más elaborados. Cada iteración añade una capa de control sobre la respuesta.

In [8]:
# The question stays the same across all iterations
USER_QUESTION = "We're migrating our payment service from a monolith to microservices. What should we consider?"

In [9]:
# Iteration 1 — Basic role: slightly better, but still unfocused
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a senior software architect with experience in distributed systems.",
    input=USER_QUESTION
)

print("=== ITERATION 1: Basic role ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 1: Basic role ===
Migrating a payment service from a monolithic architecture to microservices is a significant endeavor that requires careful planning and execution. Here are several key considerations to keep in mind:

### 1. **Identify Boundaries**

- **Domain-Driven Design (DDD):** Break down your monolith into smaller, cohesive services based on business capabilities (e.g., payment processing, fraud detection, transaction management).
- **Service Boundaries:** Define the boundaries of each microservice clearly to avoid overlap and ensure that each service has a single responsibility.

### 2. **Data Management**

- **Database per Service:** Migrate to a decentralized data model where each service manages its own database (if applicable), rather than a shared database, to avoid tight coupling.
- **Data Synchronization:** Consider how services will share data when needed. This may involve event-driven architectures (using events like Command Query Responsibility Segregat

In [10]:
# Iteration 2 — Role + audience + format constraints
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="""You are a senior software architect with 15+ years of experience in distributed systems and payment infrastructure.

Your audience is a team of experienced backend developers (5+ years) who understand software fundamentals but have never done a monolith-to-microservices migration.

Response format:
- Write in prose, no bullet points or numbered lists
- Maximum 250 words
- Prioritize actionable insights over generic theory""",
    input=USER_QUESTION
)

print("=== ITERATION 2: Role + audience + format ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 2: Role + audience + format ===
As you embark on migrating your payment service from a monolith to microservices, it's crucial to adopt a strategic and systematic approach. First, identify bounded contexts within your existing system. Focus on the parts of your payment service that can stand alone, like transaction processing, fraud detection, or user management, and think about how they interact with each other. Prioritize migrating the most critical functionalities first to minimize disruption.

Next, establish clear API contracts for each microservice. This step ensures that teams can work concurrently without waiting for each other, facilitating smoother integration down the line. Consider implementing an event-driven architecture using a message broker. This approach decouples services and allows for better scalability and fault tolerance.

Data management is another pivotal area. Aim for decentralized data ownership in your microservices; each service should have it

In [11]:
# Iteration 3 — Adding behavioral constraints and edge case handling
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="""You are a senior software architect with 15+ years of experience in distributed systems and payment infrastructure. You have led multiple monolith-to-microservices migrations at fintech companies.

Your audience is a team of experienced backend developers (5+ years) who understand software fundamentals but have never done a monolith-to-microservices migration.

Response rules:
- Write in prose, no bullet points or numbered lists
- Maximum 250 words
- Prioritize actionable insights over generic theory
- When discussing trade-offs, always present both sides with concrete consequences
- If the question lacks critical context (team size, traffic volume, regulatory requirements), explicitly state what assumptions you are making
- Never recommend a specific vendor or product — focus on architectural patterns
- Always mention at least one non-obvious risk that teams commonly overlook""",
    input=USER_QUESTION
)

print("=== ITERATION 3: Role + audience + format + constraints ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 3: Role + audience + format + constraints ===
When migrating your payment service from a monolith to microservices, consider the domain boundaries carefully. It's crucial to define clear service boundaries based on business capabilities, like transaction processing, fraud detection, and customer management. This leads to more manageable services, but be wary of potential tight coupling — if services share too much data, you risk reintroducing monolithic behavior, complicating future changes.

Another aspect revolves around data management. In a monolithic setup, the database is often a single source of truth. As you split into microservices, each service should ideally own its data store, promoting autonomy. However, this introduces the challenge of managing data consistency across services. Consider implementing sagas for distributed transactions, but know that this can complicate failure recovery and increase response times.

Infrastructure is another critical area. A m

In [30]:
# Iteration 4 — Production-grade prompt: full quality contract
PRODUCTION_SYSTEM_PROMPT = """You are a principal software architect specializing in payment systems, PCI-DSS compliance, and large-scale distributed architectures. You have led monolith-to-microservices migrations at three fintech companies processing over $1B in annual transactions.

Audience: senior backend developers (5+ years experience) planning their first major architectural migration. They understand distributed systems theory but need practical, battle-tested guidance.

Response structure:
- Open with the single most critical decision they need to make first
- Follow with 2-3 key considerations, each grounded in a concrete scenario or real-world consequence
- Close with one commonly overlooked risk and how to mitigate it
- Write in prose, no bullet points, no numbered lists, no markdown headers
- Maximum 300 words

Behavioral constraints:
- When you lack context (team size, current traffic, regulatory environment), state your assumptions explicitly before advising
- Never recommend specific vendors, cloud providers, or commercial products
- Present trade-offs with concrete consequences: 'If you choose X, expect Y; if you choose Z, expect W'
- Distinguish between what is essential on day one versus what can be deferred
- If the migration scope is ambiguous, ask one clarifying question before providing your analysis

Do not:
- Use filler phrases like 'Great question!' or 'There are many things to consider'
- Repeat information the user already provided
- Give generic advice that applies to any software project, not specifically to payment system migrations"""

response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=PRODUCTION_SYSTEM_PROMPT,
    input=USER_QUESTION
)

print("=== ITERATION 4: Production-grade prompt ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 4: Production-grade prompt ===
Your most critical initial decision revolves around the service boundaries—specifically, how to partition your payment service into microservices. Misjudging these boundaries can lead to tight coupling, complicating deployment and increasing the risk of cascading failures.

One key consideration is the transaction lifecycle. For example, if you decide to split out your payment authorization and settlement into separate services prematurely, you may face challenges in maintaining ACID properties since they often require strong consistency. If your authorization service goes down briefly, your system may over-authorize transactions, leading to potential revenue loss and compliance violations. Contrarily, if you keep these responsibilities intertwined too long, you'll miss out on scalability and deployment benefits.

Another aspect is throughput and scaling strategies. If you expect a sudden increase in transaction volume post-migration, consid

### 2.3. Abstraer al usuario del prompting

En un producto real, el usuario no escribe prompts. Solo aporta datos mínimos. El desarrollador construye el prompt internamente. Veamos dos versiones de la misma funcionalidad: una amateur y una profesional.

In [14]:
# NAIVE version — minimal prompt, inconsistent results

def review_pull_request_naive(diff_text: str) -> str:
    """Naive approach: vague instructions, unpredictable output."""
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="You are a code reviewer.",
        input=f"Review this code:\n\n{diff_text}",
        temperature=0.7
    )
    return response.output_text


sample_diff = """def process_payment(amount, currency, user_id):
    if amount > 0:
        result = gateway.charge(amount, currency)
        db.save(user_id, result)
        return result
    return None"""

print("=== NAIVE VERSION ===")
print(review_pull_request_naive(sample_diff))

=== NAIVE VERSION ===
Here's a review of the provided code snippet for the `process_payment` function:

### Code Review:

1. **Function Purpose**: The function aims to process a payment given an amount, currency, and user ID.

2. **Input Validation**:
   - The function only checks if the `amount` is greater than zero. 
   - **Improvement**: Consider validating `currency` to ensure it's a supported currency and `user_id` to verify its format or existence in your database.

3. **Error Handling**:
   - There's no error handling for the payment processing or the database operation.
   - **Improvement**: Wrap the payment processing and database saving in a try-except block to manage potential exceptions gracefully.

4. **Return Value**:
   - The function returns `None` for invalid amounts, but it might be better to raise an exception to indicate that the payment couldn't be processed due to invalid input.

5. **Logging**:
   - There’s no logging of the operation, which can be crucial for tr

In [15]:
# PRODUCTION version — detailed prompt, consistent and actionable output

def review_pull_request(diff_text: str, context: str = "") -> str:
    """
    Production-grade PR review.
    The user pastes a diff. The developer controls everything else.
    """
    instructions = """You are a staff engineer performing a pull request review for a fintech company that processes credit card payments. Your reviews are known for being thorough but respectful.

Review criteria (in priority order):
1. Security vulnerabilities: injection, authentication gaps, sensitive data exposure
2. Error handling: missing try/catch, silent failures, unhandled edge cases
3. Data integrity: race conditions, transaction boundaries, idempotency
4. Code clarity: naming, function length, single responsibility

Response format:
- Start with a one-sentence overall assessment (approve, request changes, or needs discussion)
- For each finding: describe the issue, explain the concrete risk, suggest a fix with a brief code snippet if applicable
- Maximum 3 findings — prioritize by severity, skip nitpicks
- Write in prose, no markdown headers, no bullet points

Constraints:
- Never say 'looks good' if there are security or data integrity concerns
- If the diff is too small to identify the full context, state what assumptions you are making
- Do not suggest style changes unless they affect readability in a meaningful way
- Treat the author as a competent developer — explain the 'why', not the 'how to write code'"""

    user_input = f"Pull request diff to review:\n\n```\n{diff_text}\n```"
    if context:
        user_input += f"\n\nAdditional context: {context}"

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        temperature=0.15,
        max_output_tokens=600
    )
    return response.output_text


# The user only pastes a diff — the developer controls everything else
print("=== PRODUCTION VERSION ===")
print(review_pull_request(sample_diff, context="This function runs inside our payment processing pipeline."))

print("\n" + "=" * 60)
print("The user pasted 6 lines of code.")
print("The model received a 250-word system prompt + formatted input + context.")

=== PRODUCTION VERSION ===
I request changes to this pull request due to potential security vulnerabilities and error handling issues.

First, the function lacks proper error handling for the payment processing. If the `gateway.charge` call fails (e.g., due to network issues or invalid payment details), it could lead to silent failures where the user is unaware that their payment was not processed. This could result in a poor user experience and potential financial discrepancies. To mitigate this risk, you should wrap the payment processing in a try/catch block to handle exceptions gracefully and log any errors for further investigation. For example:

```python
try:
    result = gateway.charge(amount, currency)
except Exception as e:
    log.error(f"Payment processing failed for user {user_id}: {e}")
    return None
```

Second, there is a risk of sensitive data exposure. The `result` from the `gateway.charge` call may contain sensitive information, such as transaction IDs or payment d

### 🧪 Tu turno

Crea tu propia función que abstraiga el prompting con un prompt de nivel producción. Algunas ideas:
- Analizar un requisito de software y detectar ambigüedades
- Evaluar un diseño de base de datos y sugerir mejoras
- Revisar un mensaje de error de cara al usuario y proponer uno mejor
- Analizar un endpoint API y evaluar su diseño REST

In [ ]:
# Your production-grade function here

def my_function(user_input: str) -> str:
    instructions = """..."""

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        temperature=0.2,
        max_output_tokens=500
    )
    return response.output_text

# Test:
# print(my_function("..."))

### 2.4. Consistencia de respuestas

Los modelos son estocásticos. ¿Es aceptable que dos usuarios reciban respuestas de calidad diferente ante la misma pregunta? Veamos cómo la combinación de temperature + instrucciones precisas controla la variabilidad.

In [ ]:
# High temperature + vague prompt = wildly inconsistent responses
print("=== HIGH TEMPERATURE (1.0) + VAGUE PROMPT ===")
for i in range(4):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="You are a technical consultant.",
        input="Should we use microservices?",
        temperature=1.0,
        max_output_tokens=120
    )
    # Show first 120 chars to compare at a glance
    text = response.output_text.replace("\n", " ")[:120]
    print(f"  Run {i+1}: {text}...")

=== HIGH TEMPERATURE (1.0) + VAGUE PROMPT ===
  Run 1: Deciding whether to use microservices depends on various factors related to your organization's goals, team structure, a...
  Run 2: Deciding whether to use microservices depends on various factors specific to your organization and project. Here are som...
  Run 3: Deciding whether to adopt a microservices architecture involves evaluating several factors specific to your organization...
  Run 4: Deciding whether to use microservices depends on several factors specific to your organization's needs and the nature of...


In [ ]:
# Low temperature + precise prompt = consistent, predictable responses
print("=== LOW TEMPERATURE (0.1) + PRECISE PROMPT ===")
for i in range(4):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="""You are a solutions architect. When asked about architecture decisions, always respond with exactly three sentences:
1. State the key trade-off in one sentence
2. Give the scenario where option A wins
3. Give the scenario where option B wins
Do not use bullet points. Write the three sentences as a single paragraph.""",
        input="Should we use microservices?",
        temperature=0.1,
        max_output_tokens=120
    )
    text = response.output_text.replace("\n", " ")[:120]
    print(f"  Run {i+1}: {text}...")

=== LOW TEMPERATURE (0.1) + PRECISE PROMPT ===
  Run 1: The key trade-off is between the flexibility and scalability of microservices versus the complexity and overhead they in...
  Run 2: The key trade-off is between the flexibility and scalability of microservices versus the complexity and overhead they in...
  Run 3: The key trade-off is between the flexibility and scalability of microservices versus the complexity of managing multiple...
  Run 4: The key trade-off is between the flexibility and scalability of microservices versus the complexity and overhead they in...


---
# Bloque 3 — Parámetros del modelo y herramientas integradas

**Objetivo:** Conocer los parámetros esenciales que controlan el comportamiento del modelo y las herramientas integradas que la API pone a nuestra disposición.

Parámetros que veremos:
- `temperature` — controla la aleatoriedad (con nota sobre `top_p` como alternativa)
- `max_output_tokens` — límite máximo de tokens de salida
- `stop` y `seed` — control de parada y reproducibilidad

Herramientas integradas que veremos:
- `web_search` — acceso a información actual de la web
- Mención breve de otras: `file_search`, `code_interpreter`, `computer_use`

### 3.1. Temperature: de determinista a creativo

`temperature` controla cuánta aleatoriedad introduce el modelo al elegir el siguiente token. El rango típico es 0.0 – 2.0 (en Anthropic, 0.0 – 1.0).

- **0.0** — el modelo siempre elige el token más probable. Respuestas casi deterministas.
- **0.3 – 0.5** — ligera variación. Bueno para análisis técnico, clasificación, extracción de datos.
- **0.7 – 1.0** — respuestas más variadas y creativas. Bueno para brainstorming, redacción creativa.
- **> 1.0** — alta aleatoriedad. Puede producir resultados incoherentes.

**Alternativa: `top_p` (nucleus sampling).** Limita el muestreo a los tokens cuya probabilidad acumulada supere `p`. Con `top_p=0.1`, solo se consideran los tokens más probables. **Regla:** usa `temperature` O `top_p`, nunca ambos a la vez — tanto OpenAI como Anthropic lo recomiendan explícitamente, y en modelos Claude 4.5+ su uso combinado es directamente rechazado por la API.

Veamos el efecto con una pregunta idéntica lanzada 3 veces a cada nivel:

In [10]:
TEMP_SYSTEM = "You are a product naming consultant. Propose a single product name for the given description. Respond with only the name, no explanation."
TEMP_QUESTION = "A mobile app that helps freelance developers track billable hours across multiple clients."

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    print(f"\n=== temperature = {temp} ===")
    for i in range(3):
        resp = openai_client.responses.create(
            model="gpt-4o-mini",
            instructions=TEMP_SYSTEM,
            input=TEMP_QUESTION,
            temperature=temp,
            max_output_tokens=30
        )
        print(f"  Run {i+1}: {resp.output_text.strip()}")


=== temperature = 0.0 ===
  Run 1: TimeTrack Pro
  Run 2: TimeTrack Pro
  Run 3: TimeTrack Pro

=== temperature = 0.7 ===
  Run 1: TimeSync Pro
  Run 2: TimeSync Pro
  Run 3: TimeTrackr

=== temperature = 1.5 ===
  Run 1: HourHub
  Run 2: DevHour Tracker
  Run 3: TimeNest


**Observaciones esperadas:**
- Con `temperature=0.0`, las 3 respuestas son idénticas o casi idénticas.
- Con `temperature=0.7`, las respuestas son diferentes pero todas razonables.
- Con `temperature=1.5`, las respuestas pueden ser muy creativas — o incoherentes.

**Regla práctica:** empieza siempre con `temperature=0.2-0.3` para tareas de producción. Sube solo cuando necesites creatividad explícita.

### 3.2. max_output_tokens: controlando la longitud de la respuesta

`max_output_tokens` define el **techo** de tokens que el modelo puede generar. No es la longitud objetivo — es el límite superior. Si se alcanza, la respuesta se corta abruptamente.

Dos razones para usarlo en producción:
1. **Control de coste** — los tokens de salida son 3-10x más caros que los de entrada.
2. **Prevención de respuestas runaway** — evita que el modelo genere respuestas excesivamente largas.

In [11]:
LENGTH_SYSTEM = "You are a technical writer. Explain the concept clearly and thoroughly."
LENGTH_QUESTION = "Explain how database indexing works and when to use different index types."

limits = [50, 200, 800]

for limit in limits:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=LENGTH_SYSTEM,
        input=LENGTH_QUESTION,
        temperature=0.3,
        max_output_tokens=limit
    )

    status = resp.status
    truncated = " [TRUNCATED]" if status == "incomplete" else " [COMPLETE]"

    print(f"\n=== max_output_tokens = {limit}{truncated} ===")
    print(f"Tokens generated: {resp.usage.output_tokens}")
    print(f"Response:\n{resp.output_text}")


=== max_output_tokens = 50 [TRUNCATED] ===
Tokens generated: 50
Response:
## Understanding Database Indexing

### What is Database Indexing?

Database indexing is a data structure technique that improves the speed of data retrieval operations on a database table. An index is a separate data structure that maintains a sorted order of a specific column or

=== max_output_tokens = 200 [TRUNCATED] ===
Tokens generated: 200
Response:
### Understanding Database Indexing

**Database Indexing** is a data structure technique used to quickly locate and access the data in a database table. An index is similar to an index in a book; it allows the database management system (DBMS) to find rows in a table without scanning every row.

#### How Indexing Works

1. **Structure**: An index is typically implemented as a balanced tree (like a B-tree or a B+ tree) or a hash table. These structures allow for efficient searching, insertion, and deletion operations.

2. **Creation**: When an index is created 

**Observaciones esperadas:**
- Con `max_output_tokens=50`, la respuesta se corta a mitad de frase. El `status` será `"incomplete"`.
- Con `max_output_tokens=800`, la respuesta termina de forma natural antes de llegar al límite.

**Regla práctica:** `max_output_tokens` es un límite de seguridad, no un control de longitud. Para controlar la longitud real, combínalo con instrucciones en el system prompt ("Maximum 150 words", "Respond in exactly 3 sentences").

⚠️ **Importante:** en Anthropic, `max_tokens` es obligatorio. En OpenAI y Gemini es opcional.

### 3.3. Otros parámetros útiles: stop y seed

**`stop` (OpenAI) / `stop_sequences` (Anthropic):** lista de strings. Si el modelo genera cualquiera, la generación se detiene inmediatamente. El string de parada **no** se incluye en la respuesta. Útil para garantizar un formato específico (por ejemplo, parar al encontrar `"###"` o `"END"`) o para ahorrar tokens cortando en un delimitador conocido.

**`seed`:** parámetro de OpenAI y Gemini (**no existe en Anthropic**). Misma entrada + mismo seed = misma respuesta. Imprescindible para tests automatizados que involucran LLMs — sin él, tus tests serán flaky. OpenAI no garantiza determinismo al 100% entre snapshots, pero en la práctica es muy consistente.

In [12]:
# Example 1: stop — truncate a list before the 3rd item
# NOTE: the Responses API has no `stop` parameter; it only exists in Chat Completions.
MESSAGES = [
    {"role": "system", "content": "You are a technical assistant."},
    {"role": "user", "content": "List 5 advantages of microservices architecture. Number each one."},
]

print("=== Without stop ===")
resp = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=MESSAGES,
    temperature=0.3,
    max_completion_tokens=300,
)
print(resp.choices[0].message.content)
print(f"\n[Output tokens: {resp.usage.completion_tokens} · finish_reason: {resp.choices[0].finish_reason}]")

print("\n=== With stop=['3.'] — cuts before item 3 ===")
resp = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=MESSAGES,
    temperature=0.3,
    max_completion_tokens=300,
    stop=["3."],
)
print(resp.choices[0].message.content)
print(f"\n[Output tokens: {resp.usage.completion_tokens} · finish_reason: {resp.choices[0].finish_reason}]")

=== Without stop ===
Certainly! Here are five advantages of microservices architecture:

1. **Scalability**: Microservices can be scaled independently, allowing teams to allocate resources more efficiently based on the specific needs of each service, leading to better performance and cost management.

2. **Flexibility in Technology Stack**: Different microservices can be built using different technologies and programming languages, enabling teams to choose the best tools for each specific task and facilitating the use of modern frameworks and libraries.

3. **Improved Fault Isolation**: If one microservice fails, it does not necessarily bring down the entire application. This isolation helps in maintaining overall system stability and allows for easier debugging and recovery.

4. **Faster Deployment and Development**: Smaller, independent teams can work on different microservices simultaneously, which accelerates the development process and allows for more frequent and reliable deploym

In [13]:
# Example 2: seed — reproducible outputs even with high temperature
# NOTE: the Responses API has no `seed` parameter; it only exists in Chat Completions.
SEED_MESSAGES = [
    {"role": "system", "content": "You are a creative writer."},
    {"role": "user", "content": "Write a one-sentence product tagline for a time-tracking app."},
]

print("=== Same seed (42) — reproducible output ===")
for i in range(3):
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=SEED_MESSAGES,
        temperature=0.8,
        seed=42,
        max_completion_tokens=50,
    )
    print(f"  Run {i+1}: {resp.choices[0].message.content.strip()}")

print("\n=== Different seeds — varied output ===")
for seed in [1, 100, 9999]:
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=SEED_MESSAGES,
        temperature=0.8,
        seed=seed,
        max_completion_tokens=50,
    )
    print(f"  seed={seed}: {resp.choices[0].message.content.strip()}")

=== Same seed (42) — reproducible output ===
  Run 1: "Capture every moment, maximize every minute—transform your time with our intuitive tracking app!"
  Run 2: "Capture every moment, maximize every minute—transform your time with our intuitive tracking app!"
  Run 3: "Capture every moment, maximize every minute—transform your time with our intuitive tracking app!"

=== Different seeds — varied output ===
  seed=1: "Master your minutes, reclaim your day."
  seed=100: "Master your minutes: Unlock productivity and reclaim your time with every tap!"
  seed=9999: "Master your minutes, maximize your momentum."


**Regla práctica:**
- **`stop`** es útil cuando quieres garantizar un formato específico sin depender de que el modelo "decida" cuándo parar.
- **`seed`** debería estar en todos tus tests automatizados que involucren LLMs.

### 3.4. Resumen: reglas prácticas

Configuraciones típicas por tipo de tarea:

| Tarea | `temperature` | `max_output_tokens` | `seed` |
|-------|:-------------:|:-------------------:|:------:|
| Análisis / clasificación | 0.0 – 0.2 | 200 – 500 | Sí (para tests) |
| Extracción de datos | 0.0 | 500 – 1500 | Sí |
| Resúmenes técnicos | 0.2 – 0.4 | 500 – 1000 | Opcional |
| Chat conversacional | 0.3 – 0.5 | 500 – 1500 | No |
| Redacción creativa | 0.7 – 0.9 | 1000 – 3000 | No |

No hay valores "correctos" universales — son puntos de partida que se ajustan midiendo calidad real en tu caso de uso.

⚠️ **Nota sobre modelos de razonamiento:** los modelos de razonamiento (OpenAI o-series, GPT-5 series, Claude con extended thinking) **no aceptan** la mayoría de estos parámetros. Temperature, top_p, frequency_penalty y otros están fijados internamente. En su lugar, ofrecen `reasoning_effort` (OpenAI) o `thinking.budget_tokens` (Anthropic). Tenemos una guía dedicada a este tema en el material asíncrono.

### 3.5. Herramientas integradas del modelo

Hasta ahora, nuestros LLMs solo responden con lo que saben del entrenamiento. Pero los LLMs tienen tres limitaciones fundamentales:

1. **Knowledge cutoff** — no saben nada posterior a su fecha de entrenamiento
2. **No acceden a información privada** — no pueden leer tus documentos, base de datos, etc.
3. **No ejecutan código** — no pueden calcular cosas, ejecutar SQL, procesar archivos

Las **herramientas integradas** resuelven estas limitaciones. Son funcionalidades que el proveedor ejecuta en sus propios servidores: el modelo decide cuándo usarlas, las invoca, procesa los resultados y genera la respuesta final. Todo en una sola llamada a la API desde tu lado.

**Herramientas disponibles (abril 2026):**

| Herramienta | OpenAI | Anthropic | Para qué sirve |
|-------------|:------:|:---------:|----------------|
| `web_search` | ✅ | ✅ | Buscar información actual en internet |
| `file_search` / retrieval | ✅ (con vector store) | ❌ (usa MCP) | Buscar en tus documentos |
| `code_interpreter` / `code_execution` | ✅ | ✅ | Ejecutar código Python en sandbox |
| `computer_use` | ✅ | ✅ (beta) | Controlar un ordenador (clicks, teclado) |
| `image_generation` | ✅ | ❌ | Generar imágenes |
| MCP remote servers | ✅ | ✅ | Conectar a servidores externos de herramientas |

Vamos a ver la más inmediatamente útil: `web_search`.

### 3.6. web_search con OpenAI: respuestas con información actual

La pregunta "¿qué pasó ayer en la bolsa?" es imposible de responder con el conocimiento del modelo. Con `web_search`, el modelo busca, lee páginas web relevantes, y genera una respuesta con citas.

Sintaxis en OpenAI: añades `tools=[{"type": "web_search"}]` a la llamada. El modelo decide si usar la herramienta o no.

In [14]:
# Question that requires current information
response = openai_client.responses.create(
    model="gpt-4o",
    input="What were the most important announcements at OpenAI DevDay 2025?",
    tools=[{"type": "web_search"}]
)

print("=== Response ===")
print(response.output_text)
print(f"\n=== Tokens ===")
print(f"Input: {response.usage.input_tokens} / Output: {response.usage.output_tokens}")

=== Response ===
Here’s a comprehensive, richly cited summary of the **most important announcements** from **OpenAI DevDay 2025** (held on **October 6, 2025** at Fort Mason in San Francisco):

At the opening keynote, Sam Altman shared key usage metrics: **800 million weekly ChatGPT users**, **4 million developers building on OpenAI’s platform**, and the API handling **6 billion tokens per minute**([guidetoartificialintelligence.com](https://guidetoartificialintelligence.com/en/learn/what-did-openai-announce-at-devday-2025/?utm_source=openai)).

Below are the top announcements that structured the event, grouped by theme:

– **Apps in ChatGPT & Apps SDK**  
  OpenAI unveiled a preview of the **Apps SDK**, enabling developers to embed interactive apps—like Spotify, Zillow, and Canva—directly inside ChatGPT. These apps can render UI and share conversational context via the Model Context Protocol (MCP), effectively turning ChatGPT into an app-enabled platform. OpenAI plans to launch an app 

**Citas de fuentes:** la respuesta incluye automáticamente `annotations` que referencian las URLs consultadas. Puedes acceder a ellas iterando sobre `response.output`:

In [15]:
# Extract the URLs the model consulted
print("=== Sources consulted ===")
for item in response.output:
    if item.type == "message":
        for block in item.content:
            if hasattr(block, "annotations") and block.annotations:
                for annotation in block.annotations:
                    if hasattr(annotation, "url"):
                        print(f"  - {annotation.url}")

=== Sources consulted ===
  - https://guidetoartificialintelligence.com/en/learn/what-did-openai-announce-at-devday-2025/?utm_source=openai
  - https://www.infoq.com/news/2025/10/openai-dev-day/?utm_source=openai
  - https://community.openai.com/t/devday-2025-apps-sdk-sora-2-gpt-5-pro-agentkit-new-image-generation-and-speech-to-speech-mini-models-and-more/1361279/6?utm_source=openai
  - https://venturebeat.com/ai/the-most-important-openai-announcement-you-probably-missed-at-devday-2025?utm_source=openai
  - https://venturebeat.com/ai/the-most-important-openai-announcement-you-probably-missed-at-devday-2025?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://community.openai.com/t/devday-2025-apps-sdk-sora-2-gpt-5-pro-agentkit-new-image-generation-and-speech-to-speech-mini-models-and-more/1361279/6?utm_source=openai
  - https://community.openai.com/t/devday-2025-apps-sdk-sora-2-gpt-5-pro-agentkit-new-image-generation-and-speech-to-speech-mini-models-and-m

**Observaciones importantes:**
- La API gestiona todo el flujo: decide si buscar, formula la query, lee resultados, sintetiza
- El modelo cita sus fuentes automáticamente (fundamental para productos serios)
- No todos los modelos soportan `web_search`. En el ejemplo usamos `gpt-4o`. Si pruebas con `gpt-4o-mini` puede fallar.
- Tiene coste adicional sobre los tokens normales (consulta precios actualizados en la documentación)

### 3.7. web_search con Anthropic: mismo patrón, distinta sintaxis

Anthropic ofrece la misma funcionalidad con una sintaxis ligeramente distinta. La diferencia más importante: la herramienta tiene que estar **habilitada por el administrador de la organización** en la Claude Console antes de poder usarse.

In [16]:
# Note: requires web search enabled in your organization's Claude Console
# If you get an error, the feature is not enabled on your account yet.

try:
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": "What were the most important announcements at OpenAI DevDay 2025?"
        }],
        tools=[{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 3   # Cap on number of searches
        }]
    )

    # The response may contain multiple content blocks (text + tool_use + text)
    for block in response.content:
        if block.type == "text":
            print(block.text)

    # Check how many searches were performed
    if hasattr(response.usage, "server_tool_use"):
        print(f"\n[Web searches performed: {response.usage.server_tool_use.web_search_requests}]")
    print(f"[Tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out]")

except Exception as e:
    print(f"Error: {e}")
    print("\nIf the error says web_search is not enabled, ask your org admin to enable it")
    print("in the Claude Console (Settings > Privacy > Tools).")

The most important announcements at OpenAI DevDay 2025 (held on October 6, 2025) include:

**Platform Transformation:**

ChatGPT is becoming a platform for third-party applications, with OpenAI introducing a new ChatGPT "Apps" platform that allows users to interact with external services inside ChatGPT conversations.
 
The Apps SDK is based on Model Context Protocol (MCP) and enables direct operation of apps like Spotify, Canva, Figma, Coursera, Booking.com, Zillow, and Expedia inside ChatGPT.
 
Later in 2025, OpenAI will begin reviewing and publishing third-party apps, complete with an app directory/marketplace and plans for developers to monetize their ChatGPT apps.


**Developer Tools:**

OpenAI introduced AgentKit, a dedicated toolkit and infrastructure for creating, testing, and deploying autonomous AI agents in production environments.
 
Codex, OpenAI's autonomous software engineering agent, received its general availability announcement at DevDay.


**New Models and APIs:**

Dev

**Diferencias prácticas entre ambas implementaciones:**

| Aspecto | OpenAI | Anthropic |
|---------|--------|-----------|
| Sintaxis del tool | `{"type": "web_search"}` | `{"type": "web_search_20250305", "name": "web_search"}` |
| Versionado | Implícito en el SDK | Explícito en el tool type (fecha) |
| Habilitación | Directa con API key | Requiere enable en Console |
| Control de uso | Automático | `max_uses` explícito |
| Dominios permitidos | Configurable | `allowed_domains` / `blocked_domains` |
| Pricing | Incluido en el pricing del modelo | $10 / 1.000 búsquedas + tokens |
| Citas | `annotations` en output | Citas inline en el texto |

### 3.8. Otras herramientas disponibles

No las ejecutaremos en clase, pero es importante conocerlas porque aparecerán en sesiones posteriores del programa:

**`file_search` (OpenAI) — búsqueda en tus documentos**

```python
# Requires creating a Vector Store first with your documents
response = openai_client.responses.create(
    model="gpt-4o",
    input="What is our policy on refund requests?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["<your-vector-store-id>"]
    }]
)
```

Es RAG gestionado por OpenAI: subes documentos, OpenAI los chunkea, vectoriza e indexa. El modelo busca fragmentos relevantes automáticamente. **Conectaremos con esto en el Módulo 3 cuando construyamos nuestro propio sistema RAG.**

**`code_interpreter` / `code_execution` — ejecutar código en un sandbox**

```python
# OpenAI
tools=[{"type": "code_interpreter", "container": {"type": "auto"}}]

# Anthropic
tools=[{"type": "code_execution_20250825", "name": "code_execution"}]
```

El modelo genera código Python, lo ejecuta en un entorno aislado, usa el resultado para responder. Útil para cálculos precisos, análisis de CSVs, procesamiento de archivos. **Resuelve el problema del modelo que no sabe sumar.**

**`computer_use` — controlar un ordenador**

El modelo puede ver screenshots, mover el ratón, escribir, hacer clicks. Sirve para automatizar tareas en interfaces gráficas. Todavía en beta en ambos proveedores.

**MCP (Model Context Protocol) remote servers**

Ambos proveedores soportan conectar el modelo a servidores MCP remotos. Es el estándar emergente para exponer herramientas personalizadas al modelo. **Lo veremos en el Módulo 5 cuando trabajemos con agentes.**

---

**Regla general:** las herramientas integradas transforman al LLM de "oráculo que sabe" a "agente que actúa". Son el puente entre un modelo estático y un producto realmente útil.

---
# Bloque 4 — Comparativa de proveedores en vivo

**Objetivo:** Experimentar las diferencias reales entre proveedores y entre tiers de modelos con la misma pregunta y el mismo prompt profesional.

### 4.1. Misma pregunta, distinto proveedor

Usamos un prompt de producción y una pregunta compleja para ver diferencias reales, no respuestas de juguete.

In [18]:
# Shared prompt and question for provider comparison
COMPARISON_SYSTEM_PROMPT = """You are a database architect with deep experience in both relational and NoSQL systems in production environments handling 10K+ requests per second.

When comparing technologies:
- Ground every claim in a concrete scenario (e.g., 'if your write volume exceeds X, then Y becomes a bottleneck because Z')
- Quantify trade-offs when possible (latency ranges, storage overhead percentages, operational cost implications)
- Distinguish between limitations that are fundamental vs. limitations that tooling can mitigate
- Write in prose, no bullet points, no headers
- Maximum 200 words
- Do not hedge with 'it depends' without specifying what it depends on"""

COMPARISON_QUESTION = """We have a SaaS platform with 50M rows of customer activity data growing at 2M rows/month. 
Currently on PostgreSQL 16, queries are getting slow on our analytics dashboard. 
Should we migrate the analytics workload to a columnar store or add read replicas and optimize indexes?"""

In [19]:
# OpenAI (Responses API)
openai_response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=COMPARISON_SYSTEM_PROMPT,
    input=COMPARISON_QUESTION,
    temperature=0.2,
    max_output_tokens=400
)

openai_cost = calculate_cost(
    openai_response.model,
    openai_response.usage.input_tokens,
    openai_response.usage.output_tokens
)

print("=== OpenAI (gpt-4o-mini) ===")
print(openai_response.output_text)
print(f"\nModel: {openai_response.model}")
print(f"Tokens: {openai_response.usage.input_tokens} input + {openai_response.usage.output_tokens} output")
if openai_cost:
    print(f"Cost: ${openai_cost['total']:.6f}")

=== OpenAI (gpt-4o-mini) ===
Given your current workload and growth rate, migrating the analytics workload to a columnar store like Amazon Redshift or Google BigQuery could significantly enhance query performance. Columnar databases excel in read-heavy scenarios, especially for analytical queries, as they optimize storage and retrieval by only accessing relevant columns. If your queries involve aggregations or scans over large datasets, you might see latency improvements from seconds to milliseconds, depending on the complexity of the queries.

On the other hand, adding read replicas and optimizing indexes in PostgreSQL can alleviate some pressure, but this approach has limitations. Read replicas can help distribute read traffic, but they won't fundamentally change the performance characteristics of complex analytical queries, which may still suffer from slow response times due to row-oriented storage. Index optimization can improve performance, but it typically incurs overhead in writ

In [23]:
# Anthropic (Messages API)
anthropic_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=COMPARISON_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": COMPARISON_QUESTION}],
    max_tokens=400,
)

anthropic_cost = calculate_cost(
    anthropic_response.model,
    anthropic_response.usage.input_tokens,
    anthropic_response.usage.output_tokens
)

print("=== Anthropic (claude-haiku-4-5) ===")
print(anthropic_response.content[0].text)
print(f"\nModel: {anthropic_response.model}")
print(f"Tokens: {anthropic_response.usage.input_tokens} input + {anthropic_response.usage.output_tokens} output")
if anthropic_cost:
    print(f"Cost: ${anthropic_cost['total']:.6f}")

=== Anthropic (claude-haiku-4-5) ===
At 2M rows/month, you're hitting the classic OLTP-OLAP collision. Read replicas with index optimization buys you maybe 3-6 months. Here's why: your analytics queries likely scan millions of rows across few columns (typical dashboard patterns). PostgreSQL's row-oriented storage means you're reading entire rows into memory even when you need three columns from fifty. At 50M rows with realistic row widths, this creates 200-500GB of wasted I/O per complex query.

A columnar store (DuckDB, ClickHouse, or Redshift) would compress that same data to 10-30GB, making repeated scans orders of magnitude faster. The latency difference is concrete: a typical "top customers by spend" query might take 15-30 seconds on optimized PostgreSQL but 300-800ms on columnar storage.

However, the migration friction is real. You need ETL (hourly or daily syncs), handling stale data tolerance, and operational overhead. If your analytics dashboard can tolerate 1-2 hour stalenes

In [24]:
# Side-by-side comparison
print("=" * 70)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"{'':30s} {'OpenAI':>15s}  {'Anthropic':>15s}")
print("-" * 70)
print(f"{'Input tokens':30s} {openai_response.usage.input_tokens:>15d}  {anthropic_response.usage.input_tokens:>15d}")
print(f"{'Output tokens':30s} {openai_response.usage.output_tokens:>15d}  {anthropic_response.usage.output_tokens:>15d}")
if openai_cost and anthropic_cost:
    print(f"{'Total cost':30s} {'$'+f'{openai_cost["total"]:.6f}':>15s}  {'$'+f'{anthropic_cost["total"]:.6f}':>15s}")

SIDE-BY-SIDE COMPARISON
                                        OpenAI        Anthropic
----------------------------------------------------------------------
Input tokens                               201              220
Output tokens                              215              337
Total cost                           $0.000159        $0.001905


### 4.2. Misma pregunta, distinto tier de modelo

Comparamos modelos de distintos precios dentro de OpenAI con la misma pregunta compleja. ¿Justifica la diferencia de precio la diferencia de calidad?

In [25]:
# Compare different OpenAI model tiers with a complex question
openai_models = [
    "gpt-4o-mini",
    # "gpt-5.4-mini",   # Uncomment if you have access
    # "gpt-5.4",        # Uncomment if you have access (more expensive)
]

for model_name in openai_models:
    try:
        resp = openai_client.responses.create(
            model=model_name,
            instructions=COMPARISON_SYSTEM_PROMPT,
            input=COMPARISON_QUESTION,
            temperature=0.2,
            max_output_tokens=400
        )

        cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)

        print(f"\n{'=' * 60}")
        print(f"Model: {resp.model}")
        print(f"Tokens: {resp.usage.input_tokens} input + {resp.usage.output_tokens} output")
        if cost:
            print(f"Cost: ${cost['total']:.6f}")
        print(f"\nResponse:\n{resp.output_text}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"Model {model_name}: Error — {e}")


Model: gpt-4o-mini-2024-07-18
Tokens: 201 input + 211 output
Cost: $0.000157

Response:
Given your scenario, migrating the analytics workload to a columnar store like Amazon Redshift or Google BigQuery could significantly enhance query performance. Columnar databases excel in read-heavy workloads, especially for analytical queries, as they store data in a way that optimizes for aggregation and filtering. If your current PostgreSQL setup is struggling with slow queries due to the increasing volume of data, a columnar store can reduce query latency from seconds to milliseconds, particularly for complex aggregations.

On the other hand, adding read replicas and optimizing indexes can provide some immediate relief. However, as your dataset grows, the overhead of maintaining multiple indexes can lead to increased write latency, potentially exceeding 20% in operational costs due to additional storage and compute resources. Read replicas can help distribute the load but may not fully allevia

### 🧪 Tu turno

Haz la misma comparación con modelos de Anthropic. Prueba con `claude-haiku-4-5-20251001` y si tienes acceso, con `claude-sonnet-4-6-20250514`.

In [26]:
# Compare different Anthropic model tiers
anthropic_models = [
    "claude-haiku-4-5-20251001",
    # "claude-sonnet-4-6-20250514",  # Uncomment if you have access
]

for model_name in anthropic_models:
    try:
        resp = anthropic_client.messages.create(
            model=model_name,
            system=COMPARISON_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": COMPARISON_QUESTION}],
            max_tokens=400,
            temperature=0.2
        )

        cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)

        print(f"\n{'=' * 60}")
        print(f"Model: {resp.model}")
        print(f"Tokens: {resp.usage.input_tokens} input + {resp.usage.output_tokens} output")
        if cost:
            print(f"Cost: ${cost['total']:.6f}")
        print(f"\nResponse:\n{resp.content[0].text}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"Model {model_name}: Error — {e}")


Model claude-haiku-4-5-20251001: Error — Messages.create() got an unexpected keyword argument 'temperature'


---
# Bloque 5 — Agregadores: una interfaz, múltiples proveedores

**Objetivo:** Cambiar de proveedor y modelo con una sola línea de código usando LiteLLM.

In [27]:
from litellm import completion

AGGREGATOR_SYSTEM_PROMPT = """You are a backend engineering lead evaluating infrastructure decisions.
Respond in exactly 3 sentences: the recommendation, the primary reason, and one risk to watch for.
Be specific — reference concrete thresholds, metrics, or failure modes. No generic advice."""

AGGREGATOR_QUESTION = "We process 500 webhook events per second from Stripe. Should we handle them synchronously in our Rails API or push them to a message queue first?"

# Same function, same interface — only the model string changes
providers = [
    "openai/gpt-4o-mini",
    "anthropic/claude-haiku-4-5-20251001",
]

for model_name in providers:
    try:
        resp = completion(
            model=model_name,
            messages=[
                {"role": "system", "content": AGGREGATOR_SYSTEM_PROMPT},
                {"role": "user", "content": AGGREGATOR_QUESTION}
            ],
            temperature=0.2,
            max_tokens=250
        )

        print(f"\n{'=' * 60}")
        print(f"Model: {model_name}")
        print(f"Tokens: {resp.usage.prompt_tokens} input + {resp.usage.completion_tokens} output")
        print(f"\nResponse:\n{resp.choices[0].message.content}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"{model_name}: Error — {e}")


Model: openai/gpt-4o-mini
Tokens: 89 input + 100 output

Response:
I recommend pushing the webhook events to a message queue first, such as RabbitMQ or AWS SQS, to decouple processing from the API response. This approach will help maintain a response time under 200 milliseconds for the API, ensuring that we can handle spikes in traffic without overwhelming the Rails application. One risk to watch for is message backlog; if processing fails and the queue grows beyond 10,000 messages, it could lead to increased latency and potential data loss if not monitored and managed properly.

Model: anthropic/claude-haiku-4-5-20251001
Tokens: 94 input + 197 output

Response:
**Recommendation:** Push to a message queue (SQS/RabbitMQ) immediately and process asynchronously, keeping synchronous handling only for Stripe signature validation and 202 response.

**Primary reason:** At 500 events/sec, synchronous processing will exhaust Rails connection pools (typically 5-25 connections) within seconds, c

### Observación importante

LiteLLM usa la interfaz de **Chat Completions** como formato unificado:
- `messages` en lugar de `instructions` + `input`
- `resp.choices[0].message.content` en lugar de `resp.output_text`
- `resp.usage.prompt_tokens` en lugar de `resp.usage.input_tokens`

Por eso es importante conocer ambas interfaces de OpenAI: la **Responses API** es la que usamos directamente, y **Chat Completions** es el "lenguaje común" de los agregadores.

### Wrapper con cambio de proveedor en una línea

Así se ve una función de producción que abstrae completamente el proveedor:

In [28]:
def evaluate_technical_decision(
    scenario: str,
    model: str = "openai/gpt-4o-mini"
) -> str:
    """
    Evaluate a technical decision given a scenario description.
    Switching providers = changing the model string. No other code changes.
    """
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": """You are a principal engineer performing a technical design review.
For every decision, provide: the recommended approach, one concrete reason grounded in production experience, and one thing that could go wrong.
Maximum 3 sentences. No bullet points."""},
            {"role": "user", "content": scenario}
        ],
        temperature=0.2,
        max_tokens=200
    )
    return resp.choices[0].message.content


scenario = "We want to store user session data. Considering Redis vs PostgreSQL for a system with 20K concurrent users."

# Switching providers = changing one string
print("OpenAI:")
print(evaluate_technical_decision(scenario, "openai/gpt-4o-mini"))
print()
print("Anthropic:")
print(evaluate_technical_decision(scenario, "anthropic/claude-haiku-4-5-20251001"))

OpenAI:
I recommend using Redis for storing user session data due to its in-memory data structure store, which provides extremely low latency and high throughput, essential for handling 20K concurrent users effectively. In my production experience, Redis has proven to significantly reduce response times for session retrieval compared to traditional databases, leading to a smoother user experience. However, one potential issue is that if Redis is not configured with persistence or proper backup strategies, you risk losing session data during a failure or restart.

Anthropic:
**Recommendation: Redis as primary with PostgreSQL for persistence.**

Redis handles the 20K concurrent sessions with sub-millisecond latency and built-in expiration, which we've seen reduce session management CPU overhead by 60-70% compared to database polling. The risk is data loss during Redis crashes if you don't configure AOF persistence or replication—we once lost 4 hours of session data from a single-node Red

---
# Bloque 6 — Tokens: cómo funcionan y cuánto cuestan

**Objetivo:** Entender los tokens de forma práctica — qué son, cómo se cuentan, y cómo impactan en coste.

### 6.1. Anatomía de los tokens

In [31]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

sample_texts = {
    "Short English": "How much does it cost to migrate a database?",
    "Short Spanish": "¿Cuánto cuesta migrar una base de datos?",
    "Python code": "def migrate_db(source, target):\n    return source.copy_to(target)",
    "JSON payload": '{"project": "migration", "estimate_days": 45, "confidence": "medium"}',
    "Production prompt": PRODUCTION_SYSTEM_PROMPT,  # The prompt we built in Block 2
}

print(f"{'Text':25s} {'Characters':>12s} {'Tokens':>8s} {'Tok/word':>10s}")
print("-" * 59)

for label, text in sample_texts.items():
    tokens = enc.encode(text)
    word_count = len(text.split())
    ratio = len(tokens) / word_count if word_count > 0 else 0
    print(f"{label:25s} {len(text):>12d} {len(tokens):>8d} {ratio:>9.1f}x")

Text                        Characters   Tokens   Tok/word
-----------------------------------------------------------
Short English                       44       10       1.1x
Short Spanish                       40       10       1.4x
Python code                         65       14       2.8x
JSON payload                        69       19       3.2x
Production prompt                 1560      295       1.3x


In [32]:
# Visualize how a text is decomposed into tokens
sample = "¿Cuánto cuesta migrar una base de datos PostgreSQL a Amazon Aurora?"
tokens = enc.encode(sample)

print(f"Text: {sample}")
print(f"Total: {len(tokens)} tokens\n")
print("Decomposition:")
for i, t in enumerate(tokens):
    print(f"  Token {i}: '{enc.decode([t])}' (id: {t})")

Text: ¿Cuánto cuesta migrar una base de datos PostgreSQL a Amazon Aurora?
Total: 15 tokens

Decomposition:
  Token 0: '¿Cu' (id: 102091)
  Token 1: 'ánto' (id: 110808)
  Token 2: ' cuesta' (id: 116278)
  Token 3: ' migr' (id: 16848)
  Token 4: 'ar' (id: 277)
  Token 5: ' una' (id: 1969)
  Token 6: ' base' (id: 3611)
  Token 7: ' de' (id: 334)
  Token 8: ' datos' (id: 18336)
  Token 9: ' Postgre' (id: 145223)
  Token 10: 'SQL' (id: 10430)
  Token 11: ' a' (id: 261)
  Token 12: ' Amazon' (id: 9529)
  Token 13: ' Aurora' (id: 72767)
  Token 14: '?' (id: 30)


### 🧪 Tu turno

Tokeniza textos en tu idioma, con jerga técnica de tu dominio, o el system prompt de la función que creaste en el bloque 2. ¿Cuántos tokens consume?

In [33]:
# Your text here
my_text = "..."

tokens = enc.encode(my_text)
print(f"'{my_text}' -> {len(tokens)} tokens")

'...' -> 1 tokens


### 6.2. Input vs output: la asimetría que define tu factura

In [34]:
# Experiment: same question, different output limits
TOKEN_SYSTEM_PROMPT = "You are a technical assistant. Respond in Spanish."
TOKEN_QUESTION = "Explain the CAP theorem and when each trade-off matters."

output_limits = [50, 200, 1000]

print(f"{'max_output_tokens':>20s} {'Input tok':>12s} {'Output tok':>12s} {'Cost':>12s}")
print("-" * 60)

for limit in output_limits:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=TOKEN_SYSTEM_PROMPT,
        input=TOKEN_QUESTION,
        max_output_tokens=limit,
        temperature=0.3
    )

    cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)
    cost_str = f"${cost['total']:.6f}" if cost else "N/A"

    print(f"{limit:>20d} {resp.usage.input_tokens:>12d} {resp.usage.output_tokens:>12d} {cost_str:>12s}")

   max_output_tokens    Input tok   Output tok         Cost
------------------------------------------------------------
                  50           32           50    $0.000035
                 200           32          200    $0.000125
                1000           32          459    $0.000280


In [35]:
# Scale projection: how much does this cost in production?

USERS = 10_000
CALLS_PER_USER_PER_DAY = 5
CALLS_PER_DAY = USERS * CALLS_PER_USER_PER_DAY
DAYS_PER_MONTH = 30
CALLS_PER_MONTH = CALLS_PER_DAY * DAYS_PER_MONTH

# Average call: 150 input tokens (production prompt is longer), 200 output tokens
AVG_INPUT_TOKENS = 150
AVG_OUTPUT_TOKENS = 200

models_to_compare = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "claude-haiku-4-5": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
}

print(f"Scenario: {USERS:,} users x {CALLS_PER_USER_PER_DAY} calls/day x {DAYS_PER_MONTH} days")
print(f"Total: {CALLS_PER_MONTH:,} calls/month")
print(f"Average per call: {AVG_INPUT_TOKENS} input tokens + {AVG_OUTPUT_TOKENS} output tokens")
print(f"\n{'Model':25s} {'Cost/call':>15s} {'Cost/month':>15s}")
print("-" * 58)

for model_name, prices in models_to_compare.items():
    cost_per_call = (
        (AVG_INPUT_TOKENS / 1_000_000) * prices["input"] +
        (AVG_OUTPUT_TOKENS / 1_000_000) * prices["output"]
    )
    monthly_cost = cost_per_call * CALLS_PER_MONTH
    print(f"{model_name:25s} ${cost_per_call:>14.6f} ${monthly_cost:>13.2f}")

Scenario: 10,000 users x 5 calls/day x 30 days
Total: 1,500,000 calls/month
Average per call: 150 input tokens + 200 output tokens

Model                           Cost/call      Cost/month
----------------------------------------------------------
gpt-4o-mini               $      0.000142 $       213.75
gpt-5.4-mini              $      0.001013 $      1518.75
gpt-5.4                   $      0.003375 $      5062.50
claude-haiku-4-5          $      0.001150 $      1725.00
claude-sonnet-4-6         $      0.003450 $      5175.00


### 6.3. El coste oculto de las conversaciones multi-turno

In [36]:
# Multi-turn conversation: watch input tokens grow with each turn

conversation_history = []
conversation_system_prompt = """You are a DevOps engineer helping a team debug a production incident.
Ask clarifying questions when needed. Keep each response under 3 sentences.
Focus on actionable next steps, not theory."""

questions = [
    "Our API response times spiked from 200ms to 3s about 30 minutes ago.",
    "PostgreSQL CPU is at 95%. No recent deployments.",
    "We found a query doing a sequential scan on a 40M row table. It was fine yesterday.",
    "The table was auto-vacuumed last night. Could that have invalidated the index statistics?",
    "Running ANALYZE on the table now. What else should we check while we wait?",
]

print(f"{'Turn':>6s} {'Input tok':>12s} {'Output tok':>12s} {'Cumulative cost':>18s}")
print("-" * 52)

cumulative_cost = 0

for i, question in enumerate(questions):
    conversation_history.append({"role": "user", "content": question})

    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=conversation_system_prompt,
        input=conversation_history,
        max_output_tokens=200,
        temperature=0.2
    )

    conversation_history.append({"role": "assistant", "content": resp.output_text})

    cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)
    if cost:
        cumulative_cost += cost["total"]

    print(f"{i+1:>6d} {resp.usage.input_tokens:>12d} {resp.usage.output_tokens:>12d} ${cumulative_cost:>16.6f}")

  Turn    Input tok   Output tok    Cumulative cost
----------------------------------------------------
     1           69           38 $        0.000033
     2          127           49 $        0.000082
     3          204           64 $        0.000151
     4          295           60 $        0.000231
     5          380           53 $        0.000320


In [38]:
# Visualize the cost of the system prompt across turns
system_prompt_tokens = len(enc.encode(conversation_system_prompt))

print(f"\n📈 Input token growth per turn:")
print(f"Each turn resends the entire conversation history as input tokens.")
print(f"\nSystem prompt in this example: ~{system_prompt_tokens} tokens")
print(f"The production prompt from Block 2: ~{len(enc.encode(PRODUCTION_SYSTEM_PROMPT))} tokens")
print(f"\nOver a 20-turn conversation with the production prompt:")
print(f"  System prompt alone = {len(enc.encode(PRODUCTION_SYSTEM_PROMPT))} x 20 = {len(enc.encode(PRODUCTION_SYSTEM_PROMPT)) * 20:,} input tokens")


📈 Input token growth per turn:
Each turn resends the entire conversation history as input tokens.

System prompt in this example: ~38 tokens
The production prompt from Block 2: ~295 tokens

Over a 20-turn conversation with the production prompt:
  System prompt alone = 295 x 20 = 5,900 input tokens


---
# Bloque 7 — Análisis de arquitecturas reales (discusión)

Este bloque es de discusión guiada por el instructor — no hay código que ejecutar.

**Preguntas para analizar cada producto:**

1. ¿Cómo abstrae el prompting del usuario?
2. ¿Qué modelo probablemente usa?
3. ¿Cómo gestiona los tokens?
4. ¿Usa model routing (modelos diferentes para tareas diferentes)?

**Productos sugeridos:**
- GitHub Copilot
- Asistente de soporte (Intercom / Zendesk AI)
- Herramienta de análisis de documentos (NotebookLM)

---
# Extra: Resumen de las tres interfaces API

Referencia rápida de las tres interfaces que hemos usado hoy.

In [40]:
# The three API interfaces used today, side by side:

SUMMARY_SYSTEM = """You are a senior engineer. When asked a technical question, respond with exactly one sentence that a principal engineer would find useful. No filler."""
SUMMARY_QUESTION = "What is the most common mistake teams make when adopting Kubernetes?"

# 1. OpenAI Responses API (recommended for OpenAI direct usage)
resp1 = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SUMMARY_SYSTEM,        # system prompt as top-level parameter
    input=SUMMARY_QUESTION,              # user input as string
    max_output_tokens=100,
    temperature=0.1
)
print("1. Responses API:", resp1.output_text)
print(f"   Tokens: {resp1.usage.input_tokens} in / {resp1.usage.output_tokens} out")
print(f"   Status: {resp1.status}")

print()

# 2. Anthropic Messages API
resp2 = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=SUMMARY_SYSTEM,               # system prompt as separate parameter
    messages=[{"role": "user", "content": SUMMARY_QUESTION}],  # messages array
    max_tokens=100,                      # MANDATORY in Anthropic
)
print("2. Messages API:", resp2.content[0].text)
print(f"   Tokens: {resp2.usage.input_tokens} in / {resp2.usage.output_tokens} out")
print(f"   Stop reason: {resp2.stop_reason}")

print()

# 3. LiteLLM (unified Chat Completions interface)
resp3 = completion(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "system", "content": SUMMARY_SYSTEM},   # system as message in array
        {"role": "user", "content": SUMMARY_QUESTION}
    ],
    max_tokens=100,
    temperature=0.1
)
print("3. LiteLLM:", resp3.choices[0].message.content)
print(f"   Tokens: {resp3.usage.prompt_tokens} in / {resp3.usage.completion_tokens} out")
print(f"   Finish reason: {resp3.choices[0].finish_reason}")

1. Responses API: The most common mistake teams make when adopting Kubernetes is underestimating the complexity of managing stateful applications and persistent storage.
   Tokens: 51 in / 25 out
   Status: completed

2. Messages API: Treating Kubernetes as infrastructure-as-code rather than recognizing it's fundamentally a platform that requires organizational changes in how teams think about deployment, observability, and resource management.
   Tokens: 50 in / 41 out
   Stop reason: end_turn

3. LiteLLM: The most common mistake teams make when adopting Kubernetes is underestimating the complexity of managing stateful applications and persistent storage.
   Tokens: 51 in / 24 out
   Finish reason: stop
